<a href="https://colab.research.google.com/github/kandinz/Omni-TTS/blob/main/colab_batch_json.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kiên Đoàn TTS — Giao Diện Tạo Audio & Batch JSON

> **GPU:** T4 16GB (tự động) · **Chế độ:** UI Trực quan & Batch Generation từ JSON · **Tốc độ:** Build 1 lần (~3 phút), các lần sau thay đổi text xuất audio tức thì (~3-5s/cảnh)

## 🚀 Hướng dẫn sử dụng

1. **Bước 1 & 2:** Nhấn **Runtime → Run all (Ctrl+F9)** hoặc chạy Cell 1 & Cell 2 để nạp thư viện và Model (**chỉ cần Build 1 lần**).
2. **Bước 3:** Sử dụng **Giao diện Web UI (Gradio)** để nhập Text hoặc kịch bản JSON. Thay đổi văn bản và bấm **Tạo giọng nói** để xuất audio ngay lập tức mà không cần load lại Model!

In [ ]:
# === BƯỚC 1: CÀI ĐẶT THƯ VIỆN & TẢI VOICE SAMPLE ===
import os

print('🚀 [1/3] Đang kiểm tra môi trường cài đặt...')

try:
    import omnivoice
    import gradio
    print('✅ Thư viện OmniVoice & Gradio đã được cài đặt sẵn!')
except ImportError:
    print('📦 Đang cài đặt thư viện cần thiết (~2 phút)...')
    !pip install -q omnivoice gradio "numpy<2.1" "requests==2.32.4" scipy tqdm
    !pip uninstall -y transformers
    !pip install -q "transformers>=5.3.0"
    print('✅ Cài đặt thư viện hoàn tất!')

VOICE_SAMPLE_FILE = "voice_sample.mp3"
VOICE_SAMPLE_URL = "https://raw.githubusercontent.com/kandinz/Omni-TTS/refs/heads/main/voice-sample/Minh-Quang.wav"

if not os.path.exists(VOICE_SAMPLE_FILE):
    print(f'📥 Đang tải voice sample mẫu ({VOICE_SAMPLE_FILE})...')
    !wget -q "{VOICE_SAMPLE_URL}" -O "{VOICE_SAMPLE_FILE}"
    print(f'✅ Đã tải thành công {VOICE_SAMPLE_FILE}!')
else:
    print(f'✅ File {VOICE_SAMPLE_FILE} đã sẵn sàng!')


In [ ]:
# === BƯỚC 2: KHỞI TẠO OMNIVOICE MODEL & VOICE PROMPT (BUILD 1 LẦN) ===
print('🤖 [2/3] Đang khởi động OmniVoice Model...')

import logging, time, os, re, json
import numpy as np
import torch
import scipy.io.wavfile as wavfile

# Shim: AutoFeatureExtractor removed in transformers 5.x
import transformers as _tf
class _SafeAutoFeatureExtractor:
    @staticmethod
    def from_pretrained(model_name, **kwargs):
        try:
            from transformers import AutoConfig
            cfg = AutoConfig.from_pretrained(model_name, trust_remote_code=True, **kwargs)
            sr = getattr(cfg, 'sampling_rate', 24000)
        except Exception:
            sr = 24000
        class _Result:
            sampling_rate = sr
        return _Result()
_tf.AutoFeatureExtractor = _SafeAutoFeatureExtractor

from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.utils.common import get_best_device

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Kiểm tra GPU CUDA
if torch.cuda.is_available():
    print(f'⚡ GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ CẢNH BÁO: Không tìm thấy GPU! Hãy chọn Runtime > Change runtime type > T4 GPU.')

# Nạp Model 1 lần duy nhất vào memory (globals)
DEVICE = get_best_device()
if 'model' not in globals():
    print(f'🧠 Đang nạp model OmniVoice vào {DEVICE} (float16)...')
    model = OmniVoice.from_pretrained(
        'k2-fsa/OmniVoice', device_map=DEVICE, dtype=torch.float16, load_asr=True
    )
    SAMPLING_RATE = model.sampling_rate
    print(f'✅ Model ready — Sampling Rate: {SAMPLING_RATE}Hz')
else:
    print('✅ Model đã được load sẵn trong memory từ trước!')

# Nạp Voice Clone Prompt 1 lần duy nhất
if 'VOICE_PROMPT' not in globals() or VOICE_PROMPT is None:
    print(f'🎙️ Đang khởi tạo Voice Clone Prompt từ {VOICE_SAMPLE_FILE}...')
    VOICE_PROMPT = model.create_voice_clone_prompt(ref_audio=VOICE_SAMPLE_FILE)
    print('✅ Voice Prompt ready!')
else:
    print('✅ Voice Prompt đã sẵn sàng!')

# Cấu hình sinh mặc định
GEN_CFG = OmniVoiceGenerationConfig(
    num_step=32, guidance_scale=1.8,
    denoise=True, preprocess_prompt=True, postprocess_output=True,
    position_temperature=5.0, class_temperature=0.2,
    pad_duration=0.1, fade_duration=0.1,
)


In [ ]:
# === BƯỚC 3: GIAO DIỆN WEB UI (GRADIO) — TẠO AUDIO TỨC THÌ ===
# Model đã được build từ Bước 2, tại đây chỉ cần thay đổi text và bấm Tạo để xuất audio.

import gradio as gr
import shutil

# Hàm tạo giọng nói từ 1 đoạn văn bản
def generate_single_text(text: str, speed: float):
    text = text.strip()
    if not text:
        return None, "⚠️ Vui lòng nhập văn bản cần tạo giọng nói."

    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
    if not paragraphs:
        return None, "⚠️ Văn bản không hợp lệ."

    start_t = time.time()
    with torch.inference_mode():
        if len(paragraphs) == 1:
            audio = model.generate(
                text=paragraphs[0], voice_clone_prompt=VOICE_PROMPT,
                language='vi', speed=speed, generation_config=GEN_CFG
            )[0]
        else:
            audios = []
            for i, p in enumerate(paragraphs):
                a = model.generate(
                    text=p, voice_clone_prompt=VOICE_PROMPT,
                    language='vi', speed=speed, generation_config=GEN_CFG
                )[0]
                audios.append(a)
                if i < len(paragraphs) - 1:
                    audios.append(np.zeros(int(SAMPLING_RATE * 0.3)))
            audio = np.concatenate(audios)

    waveform = (audio * 32767).astype(np.int16)
    duration = len(waveform) / SAMPLING_RATE
    gen_time = time.time() - start_t

    status_msg = f"✅ Đã tạo audio thành công! Độ dài: {duration:.1f}s | Thời gian xử lý: {gen_time:.2f}s"
    return (SAMPLING_RATE, waveform), status_msg

# Hàm tạo giọng nói hàng loạt từ JSON Script
def generate_batch_json(json_str: str, speed: float):
    try:
        scenes = json.loads(json_str)
    except Exception as e:
        return f"❌ Lỗi cú pháp JSON: {e}", None

    if not isinstance(scenes, list):
        return "❌ Dữ liệu JSON phải là danh sách (Array) dạng: [{\"filename\": \"01.wav\", \"text\": \"...\"}]", None

    output_dir = "output_audio"
    os.makedirs(output_dir, exist_ok=True)

    start_t = time.time()
    generated_count = 0

    with torch.inference_mode():
        for idx, item in enumerate(scenes):
            fname = item.get('filename', f'scene_{idx+1:02d}.wav')
            text = item.get('text', '').strip()
            if not text:
                continue

            paragraphs = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
            if len(paragraphs) == 1:
                audio = model.generate(
                    text=paragraphs[0], voice_clone_prompt=VOICE_PROMPT,
                    language='vi', speed=speed, generation_config=GEN_CFG
                )[0]
            else:
                audios = []
                for i, p in enumerate(paragraphs):
                    a = model.generate(
                        text=p, voice_clone_prompt=VOICE_PROMPT,
                        language='vi', speed=speed, generation_config=GEN_CFG
                    )[0]
                    audios.append(a)
                    if i < len(paragraphs) - 1:
                        audios.append(np.zeros(int(SAMPLING_RATE * 0.3)))
                audio = np.concatenate(audios)

            waveform = (audio * 32767).astype(np.int16)
            save_path = os.path.join(output_dir, fname)
            wavfile.write(save_path, SAMPLING_RATE, waveform)
            generated_count += 1

    # Đóng gói ZIP
    zip_filename = "output_audio.zip"
    base_zip = os.path.splitext(zip_filename)[0]
    shutil.make_archive(base_zip, "zip", output_dir)

    elapsed = time.time() - start_t
    status_msg = f"🎉 Hoàn tất sinh {generated_count}/{len(scenes)} file WAV trong {elapsed:.1f}s! Đã đóng gói file ZIP."
    return status_msg, zip_filename

# Kịch bản JSON mẫu mặc định
DEFAULT_JSON_TEXT = json.dumps([
  {
    "filename": "scene_01_intro.wav",
    "text": "4 thứ cơ bản để tạo ra một AI Agent đúng nghĩa. Vì nếu thiếu những thứ này, rất nhiều con Agent thực ra chỉ giống một chatbot được đặt tên cho hay hơn thôi. 4 thứ đó là MCP, Skill, Hook và Schedule."
  },
  {
    "filename": "scene_02_mcp_analogy.wav",
    "text": "Đầu tiên là MCP. Bạn cứ hình dung AI Agent giống như một trợ lý rất thông minh nhưng đang ngồi trong một căn phòng kín. Nếu bạn không đưa cho nó điện thoại, máy tính, tài khoản, công cụ... thì nó chỉ có thể ngồi đó trả lời câu hỏi, nó không thể thật sự làm việc với thế giới bên ngoài."
  },
  {
    "filename": "scene_03_mcp_function.wav",
    "text": "MCP sinh ra để giải quyết chuyện đó. Nó giúp AI Agent kết nối với các công cụ bên ngoài như email, Google Drive, Google Sheets, Facebook, Fanpage, YouTube, tài liệu hệ thống nội bộ hoặc những nền tảng mà bạn đang dùng trong công việc. Nó đơn giản, MCP giống như cánh tay nối dài để Agent không chỉ nói mà còn có thể làm."
  },
  {
    "filename": "scene_04_skill_concept.wav",
    "text": "Thứ hai là Skills. Skill là bộ kỹ năng bạn dạy cho Agent. Ví dụ bạn muốn AI Agent tư vấn khách hàng thì bạn phải nói rõ quy trình tư vấn của bạn là gì: gặp khách thì hỏi gì trước, phân loại nhu cầu như thế nào, khi nào giới thiệu sản phẩm, khi nào xử lý phản đối, khi nào chốt deal, khi nào chuyển cho người thật..."
  },
  {
    "filename": "scene_05_skill_value.wav",
    "text": "Tất cả những bước đó nếu bạn viết ra thành một quy trình rõ ràng thì đó chính là Skill. Nó giống như bạn tuyển một nhân sự mới, nhưng thay vì phải training từng ngày, bạn đóng gói lại cách làm việc của mình thành một bộ hướng dẫn để Agent có thể làm theo. Skill càng rõ, càng chi tiết, kết quả AI trả ra càng tốt."
  },
  {
    "filename": "scene_06_hook_rules.wav",
    "text": "Thứ ba là Hook. Hook là những quy tắc chạy ngầm. Tức là có những việc Agent phải tự kiểm tra mà không cần bạn nhắc lại mỗi lần. Ví dụ trước khi gửi email cho khách, nó phải kiểm tra xem nội dung có đúng giọng thương hiệu không, có lộ thông tin nội bộ không, có nói quá cam kết không, có vi phạm quy tắc nào bạn đã đặt ra không... Hoặc trong hệ thống kỹ thuật, trước khi chạy một hành động nào đó, nó phải kiểm tra xem hành động này có nguy hiểm cho hệ thống hay không. Hook giúp Agent làm việc an toàn hơn và đúng nguyên tắc hơn."
  },
  {
    "filename": "scene_07_schedule_automation.wav",
    "text": "Cuối cùng là tác vụ định kỳ hay Schedule. Tác vụ định kỳ thì dễ hiểu hơn, nó giống như báo thức cho Agent. Ví dụ 6 giờ sáng mỗi ngày, Agent tự thức dậy: viết bài, đăng bài viết, tìm dữ liệu mới, quét đơn hàng hôm qua, xem có bao nhiêu khách mới, bao nhiêu tin nhắn chưa trả lời, doanh thu hôm qua thế nào... rồi gửi báo cáo về cho bạn. Bạn không cần nhắc, không cần mở máy kiểm tra."
  },
  {
    "filename": "scene_08_schedule_benefit.wav",
    "text": "Nếu đã cài đúng lịch, đúng Skill, đúng dữ liệu, Agent có thể tự chạy những tác vụ lặp lại mỗi ngày. Đây là phần rất hữu ích với người kinh doanh, chủ SME, doanh nghiệp một người hoặc KOL, vì có rất nhiều việc không khó nhưng lặp lại liên tục và rất dễ quên."
  },
  {
    "filename": "scene_09_summary.wav",
    "text": "Tóm lại, để một AI Agent không chỉ là chatbot, nó cần 4 thứ: MCP để kết nối với công cụ. Skill để biết cách làm việc. Hook để tự kiểm tra và tuân thủ quy tắc. Schedule để tự chạy đúng thời điểm."
  },
  {
    "filename": "scene_10_outro.wav",
    "text": "Với tôi, AI Agent không mạnh vì cái tên \"Agent\". Nó mạnh khi được đặt trong đúng hệ thống: có công cụ, có kỹ năng, có quy tắc, có lịch vận hành. Hãy follow tôi, tôi sẽ thử cho bạn thấy cách ghép 4 thứ này lại để tạo ra một Agent cơ bản có thể dùng được trong công việc trong chuỗi series \"tăng năng xuất công việc AI Agent\""
  }
]
, ensure_ascii=False, indent=2)

print('Khởi động Giao diện Web UI...')
gr.close_all()

CSS = ".gradio-container{max-width:800px!important;margin:0 auto!important;padding:16px!important}footer{display:none!important}"
THEME = gr.themes.Soft(primary_hue='indigo')

with gr.Blocks(title='Kiên Đoàn TTS — Voice Generator', theme=THEME, css=CSS) as demo:
    gr.Markdown(
        "# 🎙️ Kiên Đoàn TTS — AI Voice Generator\n"
        "**Giọng Nam Công Nghệ** · Model đã được nạp sẵn. Nhập văn bản và bấm tạo để xuất Audio ngay lập tức!"
    )

    with gr.Tabs():
        with gr.TabItem("📝 Tạo Audio từ Text"):
            text_input = gr.Textbox(
                label="Nhập văn bản",
                lines=5,
                placeholder="Nhập đoạn văn bản bạn muốn chuyển thành giọng nói tại đây...",
                value="Chào mừng bạn đến với hệ thống tạo giọng nói AI Kiên Đoàn TTS!"
            )
            speed_single = gr.Slider(minimum=0.7, maximum=1.5, value=0.95, step=0.05, label="Tốc độ đọc (Speed)")
            btn_single = gr.Button("🚀 Tạo Giọng Nói", variant="primary")

            audio_output = gr.Audio(label="Kết quả Audio phát trực tiếp")
            status_single = gr.Textbox(label="Trạng thái", interactive=False)

            btn_single.click(
                generate_single_text,
                inputs=[text_input, speed_single],
                outputs=[audio_output, status_single]
            )

        with gr.TabItem("📦 Tạo Audio Hàng Loạt từ JSON"):
            json_input = gr.Textbox(
                label="Kịch bản JSON (Array)",
                lines=12,
                placeholder="Nhập mảng JSON danh sách các cảnh cần sinh audio...",
                value=DEFAULT_JSON_TEXT
            )
            speed_batch = gr.Slider(minimum=0.7, maximum=1.5, value=0.95, step=0.05, label="Tốc độ đọc (Speed)")
            btn_batch = gr.Button("⚡ Tạo Audio Hàng Loạt & File ZIP", variant="primary")

            status_batch = gr.Textbox(label="Trạng thái xử lý", interactive=False)
            zip_output = gr.File(label="Tải về File ZIP chứa toàn bộ Audio WAV")

            btn_batch.click(
                generate_batch_json,
                inputs=[json_input, speed_batch],
                outputs=[status_batch, zip_output]
            )

demo.launch(server_name='0.0.0.0', share=True, theme=THEME, css=CSS)


In [ ]:
# # === BƯỚC 4 (TÙY CHỌN): CHẠY BATCH BẰNG SCRIPT CELL ===
# # Nếu không muốn dùng Web UI, bạn có thể thực thi cell này để tự động nạp kịch bản từ URL/File và nén ZIP.

# import requests

# JSON_DATA_URL = "https://raw.githubusercontent.com/kandinz/Omni-TTS/refs/heads/main/voiceover-scripts/script1.json"
# JSON_FILE_PATH = "voiceover_scenes.json"

# scenes = None
# if 'JSON_DATA_URL' in globals() and JSON_DATA_URL and JSON_DATA_URL.strip():
#     try:
#         res = requests.get(JSON_DATA_URL.strip(), timeout=10)
#         if res.status_code == 200 and res.json():
#             scenes = res.json()
#             print("✅ Đã nạp kịch bản từ URL thành công!")
#     except Exception as e:
#         print(f"⚠️ Không nạp được từ URL: {e}")

# if not scenes and os.path.exists(JSON_FILE_PATH):
#     try:
#         with open(JSON_FILE_PATH, 'r', encoding='utf-8') as f:
#             scenes = json.load(f)
#             print(f"✅ Đã nạp kịch bản từ file local {JSON_FILE_PATH}!")
#     except Exception as e:
#         print(f"⚠️ Lỗi đọc file local: {e}")

# if not scenes:
#     scenes = json.loads(DEFAULT_JSON_TEXT)
#     print("ℹ️ Sử dụng kịch bản JSON mặc định.")

# print(f"🎬 Tổng số cảnh cần tạo: {len(scenes)}")
# output_dir = "output_audio"
# os.makedirs(output_dir, exist_ok=True)

# with torch.inference_mode():
#     for item in scenes:
#         fname = item['filename']
#         text = item['text']
#         save_path = os.path.join(output_dir, fname)

#         paragraphs = [p.strip() for p in re.split(r'\n\s*\n', text.strip()) if p.strip()]
#         if len(paragraphs) == 1:
#             audio = model.generate(
#                 text=paragraphs[0], voice_clone_prompt=VOICE_PROMPT,
#                 language='vi', speed=0.95, generation_config=GEN_CFG
#             )[0]
#         else:
#             audios = []
#             for i, p in enumerate(paragraphs):
#                 a = model.generate(
#                     text=p, voice_clone_prompt=VOICE_PROMPT,
#                     language='vi', speed=0.95, generation_config=GEN_CFG
#                 )[0]
#                 audios.append(a)
#                 if i < len(paragraphs) - 1:
#                     audios.append(np.zeros(int(SAMPLING_RATE * 0.3)))
#             audio = np.concatenate(audios)

#         waveform = (audio * 32767).astype(np.int16)
#         wavfile.write(save_path, SAMPLING_RATE, waveform)
#         print(f"  ✅ Đã tạo: {fname}")

# zip_path = "output_audio.zip"
# shutil.make_archive(os.path.splitext(zip_path)[0], "zip", output_dir)
# print(f"📦 Đã đóng gói xong: {zip_path}")
